<a href="https://colab.research.google.com/github/syltaer-utp/Project-S100/blob/main/S100_Equipo2_Parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S100 — Parte II. Proyecto Final
## Gasto en salud y esperanza de vida (América + Europa, 2000–2024)

**Equipo 2** · Universidad Tecnológica de Panamá  
Curso: Introducción a la Ciencia de Datos (S100)

Este cuaderno completa el ciclo de ciencia de datos: limpieza documentada, análisis exploratorio, diseño de evaluación, baseline, dos algoritmos supervisados (árbol de regresión y kNN) con ajuste de hiperparámetros y validación cruzada, tabla comparativa train/test y espacio para interpretación.

**Hipótesis (Parte I):**  
A mayor gasto en salud per cápita (USD), mayor esperanza de vida total, controlando por PIB per cápita, año y continente.

## 1. Librerías y configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Carga del dataset

Se carga `health_panel.csv` (panel país-año construido a partir de World Bank, WHO y OECD).  
Contiene indicadores de esperanza de vida, gasto en salud, mortalidad, PIB y recursos sanitarios.

In [2]:
DATA_PATH = "health_panel.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape original: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Años: {df['year'].min()} – {df['year'].max()}")
print(f"Países/entidades: {df['country_name'].nunique()}")
print(f"\nColumnas:\n{list(df.columns)}")

Shape original: 17,210 filas × 32 columnas
Años: 1960 – 2024
Países/entidades: 265

Columnas:
['country_code', 'country_name', 'year', 'life_expectancy_total', 'life_expectancy_female', 'life_expectancy_male', 'life_expectancy_gender_gap', 'le_total_yoy_change', 'region', 'income_group', 'iso2_code', 'health_spend_pct_gdp', 'health_spend_per_capita_usd', 'spend_pct_gdp_yoy_change', 'spend_per_capita_yoy_pct', 'hospital_beds_per_1000', 'physicians_per_1000', 'nurses_midwives_per_1000', 'total_health_workers_per_1000', 'infant_mortality_per_1000', 'under5_mortality_per_1000', 'maternal_mortality_per_100k', 'infant_mortality_yoy_change', 'under5_mortality_yoy_change', 'gdp_per_capita_usd', 'population_total', 'log_gdp_per_capita', 'gdp_per_capita_yoy_pct', 'le_per_gdp_point', 'le_per_1k_spend', 'le_spend_residual', 'efficiency_score']


## 3. Limpieza y construcción del subset de trabajo

**Decisiones de limpieza (justificadas):**

1. **Solo países individuales de América y Europa** — se excluyen agregados regionales ("Latin America & Caribbean", "Europe & Central Asia", etc.) para que la unidad de observación sea país-año.
2. **Años ≥ 2000** — mayor cobertura y comparabilidad de gasto en salud; reduce huecos históricos.
3. **Filas con `life_expectancy_total` y `health_spend_per_capita_usd` no nulos** — sin estas dos variables no se puede evaluar la hipótesis.
4. **Eliminación de columnas vacías, códigos ISO y variables derivadas** — no aportan información nueva o están 100 % vacías en el subset.
5. **Creación de `continent`** — variable categórica de control (Américas vs Europa).
6. **Tratamiento de faltantes en predictoras numéricas** — se eliminan filas con PIB nulo antes del split porque el porcentaje es bajo y facilita la comparación de modelos (evita imputación que podría sesgar la comparación entre algoritmos).

In [4]:
americas = [
    "Antigua and Barbuda", "Argentina", "Bahamas, The", "Barbados", "Belize",
    "Bolivia", "Brazil", "Canada", "Chile", "Colombia", "Costa Rica", "Cuba",
    "Dominica", "Dominican Republic", "Ecuador", "El Salvador", "Grenada",
    "Guatemala", "Guyana", "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "Paraguay", "Peru", "St. Kitts and Nevis",
    "St. Lucia", "St. Vincent and the Grenadines", "Suriname",
    "Trinidad and Tobago", "United States", "Uruguay", "Venezuela, RB",
]

europe = [
    "Albania", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Greece", "Hungary", "Iceland",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Moldova", "Montenegro", "Netherlands", "North Macedonia", "Norway",
    "Poland", "Portugal", "Romania", "Russian Federation", "Serbia",
    "Slovak Republic", "Slovenia", "Spain", "Sweden", "Switzerland",
    "Ukraine", "United Kingdom",
]

# Filtrar países y crear continente
df_ae = df[df["country_name"].isin(americas + europe)].copy()
df_ae["continent"] = np.where(df_ae["country_name"].isin(americas), "Americas", "Europe")

# Años >= 2000
df_ae = df_ae[df_ae["year"] >= 2000].copy()

# Quitar columnas vacías, redundantes y derivadas
cols_eliminar = [
    "region", "income_group", "iso2_code", "country_code",
    "le_per_gdp_point", "le_per_1k_spend", "le_spend_residual", "efficiency_score",
]
df_ae = df_ae.drop(columns=cols_eliminar, errors="ignore")

# Subset final: LE, gasto y PIB no nulos
df_subset = df_ae.dropna(
    subset=["life_expectancy_total", "health_spend_per_capita_usd", "gdp_per_capita_usd"]
).copy()

print(f"df_subset: {df_subset.shape[0]:,} filas × {df_subset.shape[1]} columnas")
print(f"Años: {df_subset['year'].min()} – {df_subset['year'].max()}")
print(f"Países: {df_subset['country_name'].nunique()}")
print("\nPor continente:")
print(df_subset["continent"].value_counts())
print("\nFaltantes residuales:")
print(df_subset[["life_expectancy_total", "health_spend_per_capita_usd",
                 "gdp_per_capita_usd", "year", "continent"]].isnull().sum())

df_subset: 1,786 filas × 25 columnas
Años: 2000 – 2024
Países: 75

Por continente:
continent
Europe      964
Americas    822
Name: count, dtype: int64

Faltantes residuales:
life_expectancy_total          0
health_spend_per_capita_usd    0
gdp_per_capita_usd             0
year                           0
continent                      0
dtype: int64


## 4. Definición de X e Y

- **y:** `life_expectancy_total` (continua → regresión)
- **X:** gasto en salud (hipótesis), PIB, año y continente (controles)

In [ ]:
features = [
    "health_spend_per_capita_usd",
    "gdp_per_capita_usd",
    "year",
    "continent",
]

X = df_subset[features]
y = df_subset["life_expectancy_total"]

print(f"X: {X.shape} | y: {y.shape}")
print(f"Faltantes X:\n{X.isnull().sum()}")
print(f"Faltantes y: {y.isnull().sum()}")

X: (1786, 4) | y: (1786,)
Faltantes X:
health_spend_per_capita_usd    0
gdp_per_capita_usd             0
year                           0
continent                      0
dtype: int64
Faltantes y: 0


## 5. Train / test split

- 80 % entrenamiento, 20 % prueba.  
- `random_state=42` para reproducibilidad.  
- Unidad de observación: país-año (no se agrupa por año).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test:  {X_test.shape[0]} filas")

Train: 1428 filas
Test:  358 filas


## 6. Baseline: predecir siempre la media

El baseline:

- Predice siempre el **promedio** de la esperanza de vida del conjunto de entrenamiento

Sirve como **piso de comparación**.  
Cualquier modelo útil debe tener RMSE y MAE **menores** que el baseline, y un R² claramente mayor que 0.


In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred_b = baseline.predict(X_test)

rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_b))
mae_b  = mean_absolute_error(y_test, y_pred_b)
r2_b   = r2_score(y_test, y_pred_b)

print("Baseline (media)")
print(f"  RMSE: {rmse_b:.3f}")
print(f"  MAE:  {mae_b:.3f}")
print(f"  R²:   {r2_b:.3f}")

Baseline (media)
  RMSE: 4.464
  MAE:  3.716
  R²:   -0.011


## 7. Modelo supervisado: árbol de regresión

Se elige `DecisionTreeRegressor` porque:

1. La variable objetivo es **continua** → regresión  
2. Puede capturar relaciones **no lineales** (el efecto del gasto no tiene por qué ser una recta)  
4. Permite obtener **importancia de variables** (qué tanto aporta el gasto frente al PIB, año y continente)

**Preprocesamiento:**  
`continent` es categórica → **One-Hot Encoding** (sin imponer un orden artificial).

**Hiperparámetro:**  
`max_depth=5` limita la profundidad del árbol para reducir sobreajuste.  
Es un ajuste simple; en el reporte final se puede refinar con validación cruzada.

Todo va dentro de un `Pipeline` para que el preprocesamiento y el modelo se apliquen de forma consistente y sin fuga de información.

In [ ]:
prep = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), ["continent"]),
    ("num", "passthrough", [
        "health_spend_per_capita_usd",
        "gdp_per_capita_usd",
        "year",
    ]),
])

modelo = Pipeline([
    ("prep", prep),
    ("tree", DecisionTreeRegressor(max_depth=5, random_state=42)),
])

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("Árbol de regresión (max_depth=5)")
print(f"  RMSE: {rmse:.3f}")
print(f"  MAE:  {mae:.3f}")
print(f"  R²:   {r2:.3f}")

Árbol de regresión (max_depth=5)
  RMSE: 2.458
  MAE:  1.864
  R²:   0.693


## 8. Tabla comparativa de resultados

Se reportan las **mismas tres métricas** en el conjunto de prueba:

Se observa que el árbol mejora RMSE/MAE y sube R² respecto al baseline, eso significa que esta aprendiendo algo útil a partir de las variables.

In [ ]:
tabla = pd.DataFrame({
    "Modelo": ["Baseline (media)", "Árbol de regresión"],
    "RMSE":   [rmse_b, rmse],
    "MAE":    [mae_b, mae],
    "R²":     [r2_b, r2],
})

print(tabla.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

            Modelo  RMSE   MAE     R²
  Baseline (media) 4.464 3.716 -0.011
Árbol de regresión 2.458 1.864  0.693


#Cierre del adelanto

